In [24]:
import pandas as pd
import numpy as np
import datetime as dt


In [5]:
customers_data = pd.read_csv('customers.csv')
orders_data = pd.read_csv('orders.csv')
products_data = pd.read_csv('products.csv')
ratings_data = pd.read_csv('ratings.csv')
orders_data

,order_id,customer_id,product_id,order_date,quantity
0,1,66,38,2023-08-09 08:39:23.971834,4
1,2,10,29,2023-09-08 08:39:23.971834,4
2,3,58,9,2023-07-29 08:39:23.971834,3
3,4,33,44,2023-09-13 08:39:23.971834,4
4,5,32,47,2023-07-24 08:39:23.971834,2
...,...,...,...,...,...
995,996,30,42,2023-08-15 08:39:23.971834,4
996,997,3,16,2023-09-11 08:39:23.971834,2
997,998,84,29,2023-09-05 08:39:23.971834,2
998,999,77,43,2023-08-13 08:39:23.971834,4


# change the date column to datetime format

In [7]:
orders_data['order_date'] = pd.to_datetime(orders_data['order_date'])
orders_data


,order_id,customer_id,product_id,order_date,quantity
0,1,66,38,2023-08-09 08:39:23.971834,4
1,2,10,29,2023-09-08 08:39:23.971834,4
2,3,58,9,2023-07-29 08:39:23.971834,3
3,4,33,44,2023-09-13 08:39:23.971834,4
4,5,32,47,2023-07-24 08:39:23.971834,2
...,...,...,...,...,...
995,996,30,42,2023-08-15 08:39:23.971834,4
996,997,3,16,2023-09-11 08:39:23.971834,2
997,998,84,29,2023-09-05 08:39:23.971834,2
998,999,77,43,2023-08-13 08:39:23.971834,4


# Merge dataframes to create a comprehensive dataset

In [9]:
order_product=orders_data.merge(products_data, on="product_id")
order_product_customer=order_product.merge(customers_data, on="customer_id")
order_product_customer

,order_id,customer_id,product_id,order_date,quantity,product_name,price,category,name
0,1,66,38,2023-08-09 08:39:23.971834,4,Bekant Conference Table,441,Tables & Desks,Customer_66
1,2,10,29,2023-09-08 08:39:23.971834,4,Råskog Stool,855,Chairs,Customer_10
2,3,58,9,2023-07-29 08:39:23.971834,3,Micke Desk,19,Tables & Desks,Customer_58
3,4,33,44,2023-09-13 08:39:23.971834,4,Koppang Dresser,765,Decor,Customer_33
4,5,32,47,2023-07-24 08:39:23.971834,2,Nymö Lamp Shade,157,Lighting,Customer_32
...,...,...,...,...,...,...,...,...,...
995,996,30,42,2023-08-15 08:39:23.971834,4,Småstad Wardrobe,994,Storage Solutions,Customer_30
996,997,3,16,2023-09-11 08:39:23.971834,2,Ranarp Lamp,482,Lighting,Customer_3
997,998,84,29,2023-09-05 08:39:23.971834,2,Råskog Stool,855,Chairs,Customer_84
998,999,77,43,2023-08-13 08:39:23.971834,4,Sjöpenna Floor Lamp,187,Lighting,Customer_77


# Calculating total revenue

In [10]:
product_performance = order_product_customer.groupby("product_name").agg({
    "price": "sum",
    "quantity": "sum"
}).reset_index()
product_performance

,product_name,price,quantity
0,Askvoll Wardrobe,3864,50
1,Bekant Conference Table,11907,62
2,Bernhard Chair,1280,36
3,Bestå TV Bench,18400,46
4,Billy Bookcase,5453,40
5,Brimnes Bed Storage,23190,77
6,Docksta Table,18039,54
7,Ektorp Sofa,14574,59
8,Fjälla Storage Box,18734,44
9,Fredde Desk,18854,46


In [11]:
top_products_by_revenue =product_performance.sort_values(by="price", ascending=False)
top_products_by_quantity = product_performance.sort_values(by="quantity", ascending=False)

In [12]:
top_products_by_revenue

,product_name,price,quantity
5,Brimnes Bed Storage,23190,77
41,Småstad Wardrobe,20874,52
37,Råskog Stool,20520,59
9,Fredde Desk,18854,46
8,Fjälla Storage Box,18734,44
3,Bestå TV Bench,18400,46
27,Nockeby Sofa,18101,58
6,Docksta Table,18039,54
14,Ivar Cabinet,17862,55
12,Hemnes Daybed,16900,53


In [13]:
top_products_by_quantity

,product_name,price,quantity
22,Mackapar Shoe Storage,15114,80
5,Brimnes Bed Storage,23190,77
31,Poäng Armchair,13656,63
45,Söderhamn Sofa Section,11691,63
24,Melltorp Dining Table,10332,62
1,Bekant Conference Table,11907,62
43,Strandmon Wing Chair,2050,61
49,Valje Wall Cabinet,16056,60
7,Ektorp Sofa,14574,59
37,Råskog Stool,20520,59


# Performance of products by month

In [23]:
last_month = order_product_customer[
    order_product_customer["order_date"].dt.month
    ==
    order_product_customer["order_date"].dt.month.max()]
top_clients_last_month = last_month.groupby("name").agg({
    "price": "sum"
}).reset_index()
top_clients_last_month = top_clients_last_month.sort_values(by="price", ascending=False)
top_clients_last_month

,name,price
0,Customer_1,5466
26,Customer_33,4476
38,Customer_44,4259
46,Customer_52,4236
71,Customer_76,3981
...,...,...
78,Customer_82,187
3,Customer_11,184
4,Customer_12,176
56,Customer_61,157


# RFM analysis

In [30]:
latest_date = order_product_customer["order_date"].max() + dt.timedelta(days=1)
rfm = order_product_customer.groupby("customer_id").agg({
    "order_date": lambda x: (latest_date - x.max()).days,
    "order_id": "count",
    "price": "sum"
}).rename(columns={
    "order_date": "Recency",
    "order_id": "Frequency",
    "price": "Monetary"
})
rfm["recency_score"] = pd.qcut(rfm["Recency"], 5, labels=[5, 4, 3, 2, 1])
rfm["frequency_score"] = pd.qcut(rfm["Frequency"], 5, labels=[1, 2, 3, 4, 5])
rfm["monetary_score"] = pd.qcut(rfm["Monetary"], 5, labels=[1, 2, 3, 4, 5])
rfm["RFM_Score"] = (
                    rfm["recency_score"].astype(str)
                    + rfm["frequency_score"].astype(str)
                    + rfm["monetary_score"].astype(str)
                    )
rfm_sorted = rfm.sort_values(by="RFM_Score", ascending=False)
rfm_sorted

,Recency,Frequency,Monetary,recency_score,frequency_score,monetary_score,RFM_Score
customer_id,,,,,,,
1,1,18,12106,5,5,5,555
29,1,14,9225,5,5,5,555
52,1,16,8862,5,5,5,555
47,2,14,7385,5,5,5,555
33,2,15,8769,5,5,5,555
...,...,...,...,...,...,...,...
87,21,7,3502,1,1,1,111
79,16,3,2029,1,1,1,111
46,29,4,2488,1,1,1,111


# Review

In [39]:
top_review_products  = ratings_data.groupby("product_id").size().reset_index(name="num_reviews")
top_review_products  = top_review_products.merge(products_data, on="product_id")
top_review_products  = top_review_products.sort_values(by ="num_reviews", ascending=False)
review_per_customer = ratings_data.groupby("customer_id").size()
avg_review_per_customer = review_per_customer.mean()
print(top_review_products)
print(f"Average number of reviews per customer: {avg_review_per_customer:.2f}")

    product_id  num_reviews             product_name  price           category
11          12           16           Raskog Trolley    764  Storage Solutions
37          38           14  Bekant Conference Table    441     Tables & Desks
6            7           13          Lack Side Table    717     Tables & Desks
30          31           12             Nockeby Sofa    787  Sofas & Armchairs
16          17           11   Sinnerlig Pendant Lamp    610           Lighting
5            6           11      Brimnes Bed Storage    773               Beds
27          28           11         Tarva Nightstand    547              Decor
26          27           11             Ivar Cabinet    687  Storage Solutions
8            9           11               Micke Desk     19     Tables & Desks
31          32           11             Kivik Chaise    926  Sofas & Armchairs
29          30           11     Strandmon Wing Chair     82             Chairs
44          45           11         Hektar Work Lamp

In [44]:
product_reviews = ratings_data.groupby("product_id").agg({
    "rating" : ["count", "mean"]
}).reset_index()
product_reviews.columns = ["product_id", "num_reviews", "avg_rating"]
product_reviews = product_reviews.merge(products_data, on="product_id")
product_reviews_with_average_rating = product_reviews.sort_values(by="avg_rating", ascending=False)
product_reviews_with_average_rating

,product_id,num_reviews,avg_rating,product_name,price,category
24,25,2,4.500000,Rens Sheepskin Rug,610,Decor
19,20,6,4.166667,Pjatteryd Picture,715,Decor
3,4,10,3.900000,Malm Bed Frame,202,Beds
43,44,7,3.857143,Koppang Dresser,765,Decor
21,22,5,3.800000,Nordli Chest Drawers,561,Decor
13,14,10,3.800000,Ingolf Bar Stool,609,Chairs
36,37,9,3.777778,Fredde Desk,857,Tables & Desks
46,47,4,3.750000,Nymö Lamp Shade,157,Lighting
42,43,3,3.666667,Sjöpenna Floor Lamp,187,Lighting
49,50,9,3.666667,Mörbylånga Table,298,Tables & Desks
